# AnyMatch — Full Alliance Predictions EDA

Exploratory analysis of the full-population AnyMatch predictions produced by `predict_alliance_full.py`.

The predictions file is **lean** (`PATID_A, PATID_B, pred, match_prob`). The real fields live in the records parquet, so this notebook joins back on `PATID` to:

1. Describe the `match_prob` score distribution and the predicted-match rate.
2. Derive **field-agreement flags** (SSN, DOB, name, phone, address, ...) for every pair.
3. Bucket pairs into **scenarios mirroring the synthetic-data rules** (SSN-decisive matches, name-shuffle matches, hard negatives that share a strong key, etc.) and check the model behaves sensibly per bucket — *this is our proxy for "how good", since the full population is unlabeled.*
4. Surface **concrete example pairs** at high / low / borderline confidence.

> Run this on the VM that has the data. Edit the paths in the **Config** cell.
> PHI: keep this notebook and its outputs on HIPAA-tier compute.

In [ ]:
# Windows OpenMP guard MUST precede torch/numpy/pandas import (see CLAUDE.md gotchas).
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import sys
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert os.path.exists('loo.py'), 'Run from the AnyMatch repo root (predict_alliance_full.py lives here).'
from utils.alliance_schema import id_str_dtypes, prep_record_df, serialize_set_field

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
print('ok')

## Config — edit paths to match the VM

In [ ]:
PRED_CSV       = 'data/alliance/anymatch_predictions_full.csv'
RECORDS_PARQUET = 'data/alliance/MDM_Population_cleaned_v3_2026_06_11.parquet'

THRESHOLD = 0.5          # match_prob >= THRESHOLD counted as predicted match (pred==1)
MAX_EXAMPLES = 8         # rows shown per example table

## 1. Load predictions

In [ ]:
preds = pd.read_csv(PRED_CSV, dtype={'PATID_A': 'string', 'PATID_B': 'string'})
print(f'{len(preds):,} scored pairs')
print('columns:', list(preds.columns))

# Be robust: derive pred from threshold if the column is missing/stale.
if 'pred' not in preds.columns:
    preds['pred'] = (preds['match_prob'] >= THRESHOLD).astype(int)
preds['pred_thr'] = (preds['match_prob'] >= THRESHOLD).astype(int)

# Sanity: any duplicate pairs?
dups = preds.duplicated(subset=['PATID_A', 'PATID_B']).sum()
print(f'duplicate pairs: {dups:,}')
preds.head()

## 2. Score distribution & match rate

In [ ]:
n = len(preds)
n_match = int((preds['match_prob'] >= THRESHOLD).sum())
print(f'Pairs scored      : {n:,}')
print(f'Predicted matches : {n_match:,}  ({n_match/n:.2%})')
print(f'Predicted non     : {n-n_match:,}  ({(n-n_match)/n:.2%})')
print()
print('match_prob describe:')
print(preds['match_prob'].describe())

# How confident / decisive is the model? Count pairs in the ambiguous middle band.
for lo, hi in [(0.0,0.05),(0.05,0.2),(0.2,0.4),(0.4,0.6),(0.6,0.8),(0.8,0.95),(0.95,1.01)]:
    c = ((preds['match_prob'] >= lo) & (preds['match_prob'] < hi)).sum()
    print(f'  [{lo:.2f}, {hi:.2f}) : {c:,} ({c/n:.2%})')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(preds['match_prob'], bins=50, color='steelblue', edgecolor='white')
ax[0].axvline(THRESHOLD, color='red', ls='--', label=f'threshold={THRESHOLD}')
ax[0].set(title='match_prob distribution', xlabel='match_prob', ylabel='pairs'); ax[0].legend()

# Log scale makes the U-shape (a decisive classifier piles up near 0 and 1) visible.
ax[1].hist(preds['match_prob'], bins=50, color='seagreen', edgecolor='white')
ax[1].set_yscale('log')
ax[1].axvline(THRESHOLD, color='red', ls='--')
ax[1].set(title='match_prob (log y)', xlabel='match_prob', ylabel='pairs (log)')
plt.tight_layout(); plt.show()

## 3. Join back to records & derive field-agreement flags

We load the records parquet, filter `valid_record=True`, normalize fields with the **shared schema** (so the values match what the model was scored on), and attach `*_a` / `*_b` for each pair. From those we compute boolean agreement flags used to define scenarios.

In [ ]:
records = pd.read_parquet(RECORDS_PARQUET)
for col, dt in id_str_dtypes(records.columns).items():
    records[col] = records[col].astype(dt)
if 'valid_record' in records.columns:
    records = records[records['valid_record']].copy()
records = prep_record_df(records).set_index('PATID')
print(f'{len(records):,} valid records; index unique = {records.index.is_unique}')

# Friendly feature columns we may have post-rename.
FEATS = ['first_name','middle_name','last_name','suffix','dob','ssn','ssn4',
         'sex','address','address2','city','state','zip','phone','email']
FEATS = [c for c in FEATS if c in records.columns]
records[FEATS].head()

In [ ]:
# Attach record fields for both sides of every scored pair.
left  = records[FEATS].add_suffix('_a').reindex(preds['PATID_A'].values).reset_index(drop=True)
right = records[FEATS].add_suffix('_b').reindex(preds['PATID_B'].values).reset_index(drop=True)
pairs = pd.concat([preds.reset_index(drop=True), left, right], axis=1)

missing_side = pairs[[f'{FEATS[0]}_a', f'{FEATS[0]}_b']].isna().any(axis=1).sum()
print(f'{len(pairs):,} pairs; {missing_side:,} could not be joined to a valid record on one side')
pairs = pairs.dropna(subset=[f'{FEATS[0]}_a', f'{FEATS[0]}_b']).reset_index(drop=True)
print(f'{len(pairs):,} pairs after dropping unjoinable')

In [ ]:
def norm(s):
    """Normalize a field column for agreement comparison; '' counts as missing."""
    return (s.astype('string').str.strip().str.upper()
             .replace({'': pd.NA, 'NAN': pd.NA, 'N/A': pd.NA, 'NONE': pd.NA}))

def agree(field):
    """True where both sides present and equal (after normalization)."""
    a, b = norm(pairs.get(f'{field}_a')), norm(pairs.get(f'{field}_b'))
    return a.notna() & b.notna() & (a == b)

def both_present(field):
    return norm(pairs.get(f'{field}_a')).notna() & norm(pairs.get(f'{field}_b')).notna()

def phone_overlap():
    """phone is a space-joined set; True if any token is shared."""
    def toks(s):
        return set(str(s).split()) - {'', 'nan', 'N/A'} if pd.notna(s) else set()
    a = pairs.get('phone_a'); b = pairs.get('phone_b')
    if a is None or b is None:
        return pd.Series(False, index=pairs.index)
    return pd.Series([len(toks(x) & toks(y)) > 0 for x, y in zip(a, b)], index=pairs.index)

flags = pd.DataFrame(index=pairs.index)
for f in ['first_name','middle_name','last_name','dob','ssn','ssn4','sex','address','zip','email']:
    if f in FEATS:
        flags[f'{f}_eq'] = agree(f)
flags['phone_eq'] = phone_overlap()

# Cross-field name agreement: any of A's three name fields equals any of B's
# (catches the §5.8 token-shuffle the fine-tune targets).
name_a = [norm(pairs.get(f'{f}_a')) for f in ['first_name','middle_name','last_name'] if f in FEATS]
name_b = [norm(pairs.get(f'{f}_b')) for f in ['first_name','middle_name','last_name'] if f in FEATS]
full_name_eq = agree('first_name') & agree('last_name')
name_shuffle = pd.Series(False, index=pairs.index)
for ca in name_a:
    for cb in name_b:
        name_shuffle |= (ca.notna() & cb.notna() & (ca == cb))
flags['full_name_eq'] = full_name_eq
flags['name_token_shared'] = name_shuffle

pairs = pd.concat([pairs, flags], axis=1)
flags.mean().sort_values(ascending=False).to_frame('share_of_pairs')

## 4. Scenario buckets (mirroring the synthetic-data rules)

The synthetic corpus defines positives anchored on a **surviving strong signal** (SSN, full name+DOB, phone) and negatives as **blocking-survivors that share 1–3 strong keys but are different people**. We bucket the real pairs the same way and read the model's mean `match_prob` per bucket as a sanity check:

- **SSN match** → model should strongly predict match (the fine-tune made SSN decisive).
- **Full name + DOB match** → strong match signal.
- **Name tokens shuffled but DOB matches** → the case the fine-tune targets; should still match.
- **Shares only one weak key (e.g. DOB only, or last name only)** → blocking-survivor-like; model should mostly say non-match.
- **SSN explicitly conflicts** → strong non-match signal.

In [ ]:
def ssn_conflict():
    a, b = norm(pairs.get('ssn_a')), norm(pairs.get('ssn_b'))
    return a.notna() & b.notna() & (a != b)

g = pairs  # alias
scenario = pd.Series('other', index=g.index, dtype=object)

# Order matters: assign strongest signal last so it wins.
dob_eq  = g.get('dob_eq',  pd.Series(False, index=g.index))
ssn_eq  = g.get('ssn_eq',  pd.Series(False, index=g.index))
ssn4_eq = g.get('ssn4_eq', pd.Series(False, index=g.index))
fn_eq   = g.get('full_name_eq', pd.Series(False, index=g.index))
tok     = g.get('name_token_shared', pd.Series(False, index=g.index))
phone   = g.get('phone_eq', pd.Series(False, index=g.index))

scenario[ dob_eq & ~fn_eq & ~ssn_eq ] = 'dob_only_share'
scenario[ phone & ~fn_eq & ~ssn_eq ] = 'phone_share'
scenario[ tok & dob_eq & ~fn_eq ] = 'name_shuffle+dob'
scenario[ fn_eq & ~dob_eq ] = 'fullname_only'
scenario[ fn_eq & dob_eq ] = 'fullname+dob'
scenario[ ssn4_eq & ~ssn_eq ] = 'ssn4_match'
scenario[ ssn_eq ] = 'ssn_match'
scenario[ ssn_conflict() ] = 'ssn_conflict'
g['scenario'] = scenario

summary = (g.groupby('scenario')
             .agg(n=('match_prob','size'),
                  mean_prob=('match_prob','mean'),
                  median_prob=('match_prob','median'),
                  pred_match_rate=('match_prob', lambda s: (s>=THRESHOLD).mean()))
             .sort_values('mean_prob', ascending=False))
summary['n_pct'] = (summary['n']/len(g)).map(lambda x: f'{x:.1%}')
summary.round(3)

In [ ]:
order = summary.index.tolist()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(order, summary.loc[order, 'mean_prob'], color='slateblue')
ax.axhline(THRESHOLD, color='red', ls='--', label=f'threshold={THRESHOLD}')
ax.set(ylabel='mean match_prob', title='Mean model confidence by scenario bucket')
ax.set_xticklabels(order, rotation=35, ha='right'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Direct sanity reads: SSN-match pairs should mostly be predicted matches;
# weak-single-key pairs should mostly be non-matches.
def rate(mask, name):
    m = mask & pairs['match_prob'].notna()
    if m.sum() == 0:
        print(f'{name:28s}: (no pairs)'); return
    print(f'{name:28s}: n={m.sum():>8,}  pred-match={ (pairs.loc[m,"match_prob"]>=THRESHOLD).mean():.1%}  mean_prob={pairs.loc[m,"match_prob"].mean():.3f}')

rate(ssn_eq, 'SSN matches')
rate(fn_eq & dob_eq, 'Full name + DOB match')
rate(tok & dob_eq & ~fn_eq, 'Name shuffle + DOB')
rate(dob_eq & ~fn_eq & ~ssn_eq, 'DOB-only share')
rate(ssn_conflict(), 'SSN conflict (diff SSNs)')

## 5. Concrete example pairs

Side-by-side views so you can eyeball whether the model's call is reasonable. Helper renders the key fields for both sides plus the score.

In [ ]:
SHOW = [f for f in ['first_name','middle_name','last_name','dob','ssn','ssn4','sex','address','city','zip','phone','email'] if f in FEATS]

def show_pairs(df, k=MAX_EXAMPLES):
    """Render up to k pairs as stacked A/B rows with the score."""
    rows = []
    for _, r in df.head(k).iterrows():
        rows.append({'side':'A','PATID':r['PATID_A'], 'match_prob':round(r['match_prob'],4),
                     'scenario':r.get('scenario',''), **{f:r.get(f'{f}_a') for f in SHOW}})
        rows.append({'side':'B','PATID':r['PATID_B'], 'match_prob':'',
                     'scenario':'', **{f:r.get(f'{f}_b') for f in SHOW}})
        rows.append({'side':'', 'PATID':''})  # spacer
    return pd.DataFrame(rows).fillna('')

In [ ]:
print('=== Highest-confidence predicted MATCHES ===')
show_pairs(pairs.sort_values('match_prob', ascending=False))

In [ ]:
print('=== Borderline pairs (match_prob near the threshold) ===')
border = pairs.assign(dist=(pairs['match_prob']-THRESHOLD).abs()).sort_values('dist')
show_pairs(border)

In [ ]:
print('=== Suspicious: predicted MATCH but SSNs conflict (potential false positives) ===')
fp = pairs[(pairs['match_prob']>=THRESHOLD) & ssn_conflict()].sort_values('match_prob', ascending=False)
print(f'{len(fp):,} such pairs')
show_pairs(fp)

In [ ]:
print('=== Suspicious: predicted NON-match but SSN matches (potential false negatives) ===')
fn_susp = pairs[(pairs['match_prob']<THRESHOLD) & ssn_eq].sort_values('match_prob')
print(f'{len(fn_susp):,} such pairs')
show_pairs(fn_susp)

In [ ]:
print('=== Name-shuffle matches the fine-tune targets (shared name tokens + DOB, predicted match) ===')
ns = pairs[(pairs['match_prob']>=THRESHOLD) & tok & dob_eq & ~fn_eq].sort_values('match_prob', ascending=False)
print(f'{len(ns):,} such pairs')
show_pairs(ns)

## 6. Per-record match degree (optional)

How many predicted matches each record participates in — large clusters can indicate either a prolific duplicate or an over-merging failure mode worth inspecting.

In [ ]:
m = pairs[pairs['match_prob'] >= THRESHOLD]
deg = pd.concat([m['PATID_A'], m['PATID_B']]).value_counts()
print(f'records in >=1 predicted match: {deg.size:,}')
print('match-degree distribution:')
print(deg.describe())
print('\ntop-15 most-matched records (inspect for over-merging):')
deg.head(15)

## 7. Takeaways

Fill in after running:

- Overall predicted-match rate: ____
- Score distribution decisive (U-shaped) or muddy?  ____
- SSN-match pred-match rate (want ~high): ____
- DOB-only-share pred-match rate (want low): ____
- Any false-positive pattern (SSN-conflict matches)?  ____
- Name-shuffle cases handled?  ____